# <font color="steelblue">01. Estadística para contar historias: Colab y primeros datos</font>

*Unidad 1. Introducción* · Sesión teórica 1 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`encuesta_medios.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["encuesta_medios.csv"]
PREGUNTAS = ["q_P01_1.json", "q_P01_2.json", "q_P01_3.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

Casi cualquier noticia lleva números: el paro baja un 2 %, la mitad de los jóvenes se informa por redes, una encuesta da empate técnico. Un periodista que no sabe leer esas cifras **las repite**; uno que sabe, **las comprueba, las explica y encuentra la historia que esconden**.

La Estadística es la herramienta para eso: resumir muchos datos en pocas cifras honestas, ver si una diferencia es real o casualidad, y medir cuánto nos podemos fiar de una muestra. En este curso la aprenderemos haciendo **periodismo de datos** con Python, en cuadernos como este.

**Qué necesitas saber antes**: nada. Este es el primer cuaderno.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Explicar para qué sirve la Estadística en el trabajo periodístico.
2. Usar Google Colab: celdas de texto y de código, ejecutar y guardar.
3. Cargar una tabla de datos y obtener tu primer recuento y tu primer gráfico.

## <font color='steelblue'>Contenidos</font>

1. Por qué un periodista necesita Estadística
2. Google Colab en cinco minutos
3. Tu primer análisis: ¿por dónde se informa la gente?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Por qué un periodista necesita Estadística</font>

Mira estos titulares inventados, pero muy parecidos a los reales:

| Titular | Qué hay que preguntar |
| :-- | :-- |
| «El salario medio de los periodistas supera los 29.000 euros» | ¿Media o mediana? Unos pocos sueldos altos suben mucho la media. |
| «Los delitos se disparan un 50 %» | ¿De 2 a 3 casos o de 2.000 a 3.000? ¿Porcentaje o puntos? |
| «El partido A gana: 34 % frente a 33 %» | Con 1.000 entrevistas el margen de error es de unos ±3 puntos: es un empate. |
| «Quien lee prensa vive más» | ¿Causa o simple asociación? Quizá ambas cosas dependen de la renta. |

Cada fila es un tema de este curso: **medidas de centralización**, **tasas de variación y números índice**, **margen de error** y **correlación frente a causalidad**.

La guía docente organiza la asignatura en cinco unidades, y estos 30 cuadernos las siguen en orden:

1. **Introducción** (cuadernos 1–4): Colab, Python mínimo y tablas de datos.
2. **Estadística descriptiva** (5–13): resumir y representar datos.
3. **Números índice y series cronológicas** (14–19): IPC, términos reales, tendencias.
4. **Probabilidad** (20–24): azar, probabilidad condicionada, binomial y normal.
5. **Inferencia** (25–30): muestras, encuestas, margen de error y contrastes.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P01_1.json")

## <font color='steelblue'>2. Google Colab en cinco minutos</font>

Un **cuaderno** mezcla dos tipos de celdas:

- **Celdas de texto** (como esta): explicaciones con formato. Haz doble clic para editarlas.
- **Celdas de código**: instrucciones en Python. Se ejecutan con el botón ▶ o con **Mayús + Intro**, y el resultado aparece debajo.

Colab se ejecuta en un ordenador de Google: no hay que instalar nada. Para guardar tus cambios usa **Archivo → Guardar una copia en Drive**.

**Qué vamos a hacer**: ejecutar nuestra primera celda de código. Python puede funcionar como calculadora.

In [ ]:
# Esto es un comentario: Python ignora todo lo que va detrás de #
3200 / 4000 * 100

**Qué ha pasado**: Python ha calculado qué porcentaje son 3.200 lectores de un total de 4.000: un 80 %. La última línea de una celda se muestra siempre como resultado.

Con `print()` podemos mostrar varios resultados y añadir texto:

In [ ]:
print("Lectores de pago:", 3200)
print("Porcentaje:", 3200 / 4000 * 100, "%")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y marca **todas** las respuestas correctas:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P01_2.json")

## <font color='steelblue'>3. Tu primer análisis: ¿por dónde se informa la gente?</font>

Vamos a usar una **encuesta simulada** a 1.200 personas de la Comunitat Valenciana sobre cómo consumen noticias. Cada **fila** es una persona entrevistada y cada **columna** una pregunta.

**Qué vamos a hacer**: leer el fichero con la librería *Pandas* y ver las cinco primeras filas.

In [ ]:
encuesta = pd.read_csv("encuesta_medios.csv")
encuesta.head()

**Qué ha pasado**: `pd.read_csv()` ha leído el fichero y lo ha guardado con el nombre `encuesta`. `head()` muestra las primeras filas. Ya verás estas funciones con calma en el cuaderno 4.

Ahora contamos cuántas personas eligen cada medio como fuente principal, con `value_counts()`:

In [ ]:
encuesta["medio_principal"].value_counts()

Y lo convertimos en un gráfico de barras, que es como mejor se lee un recuento:

In [ ]:
recuento = encuesta["medio_principal"].value_counts()
recuento.plot(kind="barh", color="steelblue")
plt.title("Medio principal para informarse (encuesta simulada, n = 1.200)")
plt.xlabel("Número de personas")
plt.show()

**Cómo leerlo**: la televisión sigue siendo el medio más citado, seguida de la prensa digital y las redes sociales. Fíjate en que el título del gráfico dice **qué** se mide, **de dónde** sale y **cuántas** personas: es lo mínimo que debe llevar cualquier gráfico que publiques.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P01_3.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 2)*

Calcula con Python qué porcentaje representan 264 personas sobre un total de 1.200.

<details><summary>Ver solución</summary>

```python
264 / 1200 * 100
```
</details>

### **Ejercicio 2** *(sección 3)*

Cuenta cuántas personas hay de cada provincia en la encuesta.

<details><summary>Ver solución</summary>

```python
encuesta["provincia"].value_counts()
```
</details>

### **Ejercicio 3** *(sección 3)*

Haz un gráfico de barras horizontales con el nivel de estudios de las personas entrevistadas. Ponle un título que diga qué se mide y de dónde salen los datos.

<details><summary>Ver solución</summary>

```python
encuesta["estudios"].value_counts().plot(kind="barh", color="steelblue")
plt.title("Nivel de estudios (encuesta simulada, n = 1.200)")
plt.show()
```
</details>

### **Ejercicio 4** *(sección 1)*

Escribe en una celda de texto (botón **+ Texto**) un titular honesto con el resultado del gráfico de medios, y debajo una frase que diga qué precaución tendría que mencionar la noticia.

<details><summary>Ver idea clave</summary>

Por ejemplo: «La televisión sigue siendo la primera fuente de noticias para más de un tercio de la población». Precaución: es una encuesta (una muestra), así que las cifras tienen margen de error; y aquí, además, los datos son simulados.
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- La Estadística sirve al periodista para **comprobar, resumir y explicar** cifras, y para no repetir errores ajenos.
- Un cuaderno de Colab combina **texto** y **código**; el código se ejecuta con ▶ o Mayús + Intro.
- En una tabla de datos, **cada fila es un caso** (una persona) y **cada columna una variable** (una pregunta).
- `value_counts()` cuenta cuántas veces aparece cada valor; `plot(kind="barh")` lo dibuja.
- Todo gráfico publicado debe decir **qué se mide, de dónde salen los datos y cuántos casos hay**.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Cuaderno (notebook)** | Documento que mezcla texto y código ejecutable. |
| **Celda** | Bloque de texto o de código dentro de un cuaderno. |
| **Caso** | Cada unidad sobre la que se recogen datos (una persona, un municipio, un día). |
| **Variable** | Cada característica que se mide de los casos (edad, medio principal...). |
| **Periodismo de datos** | Periodismo que obtiene, analiza y visualiza datos para contar historias. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 2 aprenderás lo mínimo de Python que necesitas: números, textos, listas y condiciones.

**Siguiente cuaderno**: [02. Python mínimo (I): números, textos, listas y condiciones](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P02_python_minimo_1.ipynb)